# YOLO26 Scratch Segmentation Training

This notebook trains a YOLO segmentation model from the converted dataset.ab before running the cells.

## 1. Install Ultralytics

In [ ]:
"""
    Install and import YOLO training dependencies.

    Main package:
        ultralytics: provides YOLO segmentation training, validation, and export APIs.

    Utility imports:
        Path    : filesystem paths
        shutil  : optional file/folder copying
        zipfile : dataset zip extraction
        yaml    : inspect data.yaml
"""
!python -m pip install -q -U ultralytics

from pathlib import Path
import shutil
import zipfile

import yaml
from ultralytics import YOLO

print('Ultralytics is ready')


In [ ]:
"""
    Prepare scratch_yolo_seg under /content before dataset settings.

    Priority:
        1. Use existing /content/scratch_yolo_seg if available.
        2. Mount Google Drive and copy dataset folder to /content.
        3. Extract scratch_yolo_seg.zip from Drive to /content.
        4. If running in Colab and nothing is found, ask for manual zip upload.

    Expected result:
        /content/scratch_yolo_seg/data.yaml
        /content/scratch_yolo_seg/train/images
        /content/scratch_yolo_seg/valid/images
        /content/scratch_yolo_seg/test/images
"""
from pathlib import Path
import shutil
import zipfile

DATASET_NAME = 'scratch_yolo_seg'
CONTENT_ROOT = Path('/content')
CONTENT_DATASET_ROOT = CONTENT_ROOT / DATASET_NAME
DRIVE_PROJECT = Path('/content/drive/MyDrive/Surface-Scratch-Detection')
DRIVE_DATA_DIR = DRIVE_PROJECT / 'data'


def is_yolo_dataset(path: Path) -> bool:
    """
        Check whether a folder looks like a YOLO segmentation dataset.
    """
    return (
        path.is_dir()
        and (path / 'data.yaml').is_file()
        and (path / 'train' / 'images').is_dir()
        and (path / 'valid' / 'images').is_dir()
        and (path / 'test' / 'images').is_dir()
    )


def extract_dataset_zip(zip_path: Path, dst_root: Path) -> Path:
    """
        Extract a dataset zip into /content and return the dataset root.
    """
    print(f'Extracting {zip_path} -> {dst_root}')
    with zipfile.ZipFile(zip_path, 'r') as archive:
        archive.extractall(dst_root)

    direct_root = dst_root / DATASET_NAME
    if is_yolo_dataset(direct_root):
        return direct_root

    candidates = [
        path for path in dst_root.rglob('data.yaml')
        if is_yolo_dataset(path.parent)
    ]
    if candidates:
        found_root = candidates[0].parent
        if found_root != direct_root:
            shutil.copytree(found_root, direct_root, dirs_exist_ok=True)
        return direct_root

    raise FileNotFoundError(f'Could not find YOLO data.yaml after extracting {zip_path}')


def copy_dataset_to_content() -> Path:
    """
        Copy or extract scratch_yolo_seg to /content.

        Returns:
            dataset_root: /content/scratch_yolo_seg
    """
    if is_yolo_dataset(CONTENT_DATASET_ROOT):
        print('Dataset already exists:', CONTENT_DATASET_ROOT)
        return CONTENT_DATASET_ROOT

    try:
        from google.colab import drive, files
        in_colab = True
    except ModuleNotFoundError:
        drive = None
        files = None
        in_colab = False

    if in_colab:
        drive.mount('/content/drive')

    drive_dataset_dir = DRIVE_DATA_DIR / DATASET_NAME
    drive_dataset_zip = DRIVE_DATA_DIR / f'{DATASET_NAME}.zip'
    project_dataset_zip = DRIVE_PROJECT / f'{DATASET_NAME}.zip'

    if is_yolo_dataset(drive_dataset_dir):
        print(f'Copying {drive_dataset_dir} -> {CONTENT_DATASET_ROOT}')
        shutil.copytree(drive_dataset_dir, CONTENT_DATASET_ROOT, dirs_exist_ok=True)
        return CONTENT_DATASET_ROOT

    for zip_path in [drive_dataset_zip, project_dataset_zip, CONTENT_ROOT / f'{DATASET_NAME}.zip']:
        if zip_path.is_file():
            return extract_dataset_zip(zip_path, CONTENT_ROOT)

    if in_colab:
        print('Dataset not found in Drive. Upload scratch_yolo_seg.zip now.')
        uploaded = files.upload()
        for file_name in uploaded.keys():
            uploaded_path = CONTENT_ROOT / file_name
            if uploaded_path.suffix.lower() == '.zip':
                return extract_dataset_zip(uploaded_path, CONTENT_ROOT)

    raise FileNotFoundError(
        'Dataset not found. Put scratch_yolo_seg/ or scratch_yolo_seg.zip in '
        '/content/drive/MyDrive/Surface-Scratch-Detection/data, or upload the zip.'
    )


CONTENT_DATASET_ROOT = copy_dataset_to_content()
print('Ready dataset root:', CONTENT_DATASET_ROOT)


## 2. Dataset Settings

In [ ]:
"""
    Locate the YOLO instance-segmentation dataset.

    Dataset name:
        scratch_yolo_seg

    Accepted locations:
        /content/scratch_yolo_seg/
        /content/scratch_yolo_seg.zip
        /content/data/scratch_yolo_seg/
        /content/data/scratch_yolo_seg.zip
        /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_yolo_seg/
        /content/drive/MyDrive/Surface-Scratch-Detection/data/scratch_yolo_seg.zip

    Expected YOLO layout:
        train/images, train/labels
        valid/images, valid/labels
        test/images,  test/labels
        data.yaml
"""
DATASET_NAME = 'scratch_yolo_seg'

candidate_dirs = [
    Path('/content') / DATASET_NAME,
    Path('/content/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / DATASET_NAME,
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / DATASET_NAME,
]

candidate_zips = [
    Path('/content') / f'{DATASET_NAME}.zip',
    Path('/content/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection/data') / f'{DATASET_NAME}.zip',
    Path('/content/drive/MyDrive/Surface-Scratch-Detection') / f'{DATASET_NAME}.zip',
]

def find_dataset_root() -> Path:
    """
        Find a YOLO dataset folder or extract a YOLO dataset zip.

        Returns:
            Path to the folder containing data.yaml.
    """
    for path in candidate_dirs:
        if path.is_dir():
            return path

    for zip_path in candidate_zips:
        if not zip_path.is_file():
            continue

        extract_root = Path('/content')
        print(f'Extracting {zip_path} -> {extract_root}')
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(extract_root)

        for path in candidate_dirs:
            if path.is_dir():
                return path

    raise FileNotFoundError(
        'Dataset not found. Upload scratch_yolo_seg/ or scratch_yolo_seg.zip to /content.'
    )

DATASET_ROOT = find_dataset_root()
DATA_YAML = DATASET_ROOT / 'data.yaml'

print('Dataset root:', DATASET_ROOT)
print('data.yaml:', DATA_YAML)


## 3. Check YOLO Dataset Structure

In [ ]:
"""
    Validate the YOLO segmentation dataset before training.

    Checks:
        - train/valid/test image and label folders exist.
        - data.yaml exists.
        - every image has a matching .txt label file.
        - print how many labels are positive/non-empty.

    Colab safety:
        The exported data.yaml may contain an absolute local path from the PC.
        This cell writes data_colab.yaml with path rebased to DATASET_ROOT.

    Note:
        Empty label files are valid for negative images, but missing label files are not.
"""
required_dirs = [
    DATASET_ROOT / 'train' / 'images',
    DATASET_ROOT / 'train' / 'labels',
    DATASET_ROOT / 'valid' / 'images',
    DATASET_ROOT / 'valid' / 'labels',
    DATASET_ROOT / 'test' / 'images',
    DATASET_ROOT / 'test' / 'labels',
]

missing = [path for path in required_dirs if not path.is_dir()]
if missing:
    raise FileNotFoundError(
        'Missing dataset folders:\n' + '\n'.join(str(path) for path in missing)
    )
if not DATA_YAML.is_file():
    raise FileNotFoundError(f'data.yaml not found: {DATA_YAML}')

for split in ('train', 'valid', 'test'):
    image_dir = DATASET_ROOT / split / 'images'
    label_dir = DATASET_ROOT / split / 'labels'
    images = sorted([
        path for path in image_dir.iterdir()
        if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    ])
    labels = sorted(label_dir.glob('*.txt'))
    missing_labels = [
        path.name for path in images
        if not (label_dir / f'{path.stem}.txt').is_file()
    ]
    positive_labels = [path for path in labels if path.stat().st_size > 0]

    print(
        f'{split}: images={len(images)} labels={len(labels)} '
        f'positive_labels={len(positive_labels)} missing_labels={len(missing_labels)}'
    )

    if missing_labels:
        raise RuntimeError(f'{split} has missing labels. Example: {missing_labels[:5]}')

with DATA_YAML.open('r', encoding='utf-8') as file:
    data_config = yaml.safe_load(file)

"""Rebase dataset paths for the current Colab/runtime location."""
data_config['path'] = str(DATASET_ROOT)
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'
data_config['test'] = 'test/images'

RUNTIME_DATA_YAML = DATASET_ROOT / 'data_colab.yaml'
with RUNTIME_DATA_YAML.open('w', encoding='utf-8') as file:
    yaml.safe_dump(data_config, file, sort_keys=False)

DATA_YAML = RUNTIME_DATA_YAML

print('\nRuntime data.yaml:')
print(yaml.safe_dump(data_config, sort_keys=False))
print('Using data yaml:', DATA_YAML)


## 4. Train YOLO26 Segmentation

In [ ]:
"""
    Train YOLO26 segmentation for scratch instance segmentation.

    Model:
        MODEL = yolo26s-seg.pt
            s model is the recommended quality baseline for the current
            scratch_yolo_seg dataset. It has more capacity than n while still
            being practical for Colab GPU training.

    Training settings:
        EPOCHS   : 150, gives YOLO26s more room to converge.
        IMGSZ    : 512, matches current YOLO patch dataset size.
        BATCH    : 8, safer for YOLO26s on Colab T4/L4 memory.
        PATIENCE : 40 epochs without improvement before early stopping.
        LR0      : 1e-3, conservative fine-tuning learning rate.
        WORKERS  : 2 dataloader workers for Colab stability.

    Output:
        /content/yolo_scratch_runs/scratch_yolo26s_seg_v2/weights/best.pt
        /content/yolo_scratch_runs/scratch_yolo26s_seg_v2/weights/last.pt
"""
MODEL = 'yolo26s-seg.pt'
PROJECT = '/content/yolo_scratch_runs'
RUN_NAME = 'scratch_yolo26s_seg_v2'

EPOCHS = 150
IMGSZ = 512
BATCH = 8
PATIENCE = 40
LR0 = 1.0e-3
WORKERS = 2
SEED = 42
DEVICE = 0

"""Load pretrained YOLO segmentation checkpoint."""
model = YOLO(MODEL)

"""Start Ultralytics segmentation training."""
results = model.train(
    data=str(DATA_YAML),
    task='segment',
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    workers=WORKERS,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=False,
    pretrained=True,
    plots=True,
    save=True,
    device=DEVICE,
    seed=SEED,
    lr0=LR0,
)

"""Resolve important run artifact paths for later cells."""
RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = RUN_DIR / 'weights' / 'best.pt'
LAST_PT = RUN_DIR / 'weights' / 'last.pt'

print('Run dir:', RUN_DIR)
print('Best checkpoint:', BEST_PT, BEST_PT.exists())
print('Last checkpoint:', LAST_PT, LAST_PT.exists())


## 5. Validate Best Checkpoint

In [ ]:
"""
    Validate the best YOLO checkpoint on the test split.

    Input:
        BEST_PT from the training run.

    Metrics:
        Ultralytics prints segmentation metrics such as precision, recall,
        mAP, and mask metrics for the configured test split.

    Output:
        Validation plots/metrics are saved under the YOLO run directory.
"""
best_model = YOLO(str(BEST_PT))
metrics = best_model.val(
    data=str(DATA_YAML),
    task='segment',
    split='test',
    imgsz=IMGSZ,
    batch=BATCH,
    plots=True,
    device=DEVICE,
)

print(metrics)


## 6. Download Checkpoints

In [ ]:
"""
    Download YOLO checkpoints from Colab.

    Files:
        best.pt : best validation checkpoint, usually used for inference/export.
        last.pt : last training checkpoint, useful for resume/debugging.
"""
from google.colab import files

if BEST_PT.is_file():
    files.download(str(BEST_PT))
if LAST_PT.is_file():
    files.download(str(LAST_PT))


## 7. Save YOLO Run Folder to Drive


In [ ]:
"""
    Save the full YOLO training run folder to Google Drive.

    Source:
        /content/yolo_scratch_runs

    Destination:
        /content/drive/MyDrive/Surface-Scratch-Detection/models/yolo/yolo_scratch_runs

    Saved artifacts:
        - weights/best.pt
        - weights/last.pt
        - results.csv
        - args.yaml
        - metric plots and confusion matrices

    Note:
        This copies/merges files into Drive without deleting older runs already
        stored there.
"""
from pathlib import Path
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is not available. Run this cell in Colab.')

DRIVE_PROJECT = Path('/content/drive/MyDrive/Surface-Scratch-Detection')
DRIVE_YOLO_DIR = DRIVE_PROJECT / 'models' / 'yolo'
LOCAL_RUNS_DIR = Path(PROJECT)
DRIVE_RUNS_DIR = DRIVE_YOLO_DIR / LOCAL_RUNS_DIR.name

if not LOCAL_RUNS_DIR.is_dir():
    raise FileNotFoundError(f'YOLO runs folder not found: {LOCAL_RUNS_DIR}')

DRIVE_RUNS_DIR.parent.mkdir(parents=True, exist_ok=True)

"""Copy the complete yolo_scratch_runs folder to Drive."""
shutil.copytree(
    LOCAL_RUNS_DIR,
    DRIVE_RUNS_DIR,
    dirs_exist_ok=True,
)

DRIVE_CURRENT_RUN_DIR = DRIVE_RUNS_DIR / RUN_DIR.name
DRIVE_BEST_PT = DRIVE_CURRENT_RUN_DIR / 'weights' / 'best.pt'
DRIVE_LAST_PT = DRIVE_CURRENT_RUN_DIR / 'weights' / 'last.pt'

print('Saved YOLO runs folder to Drive:')
print(DRIVE_RUNS_DIR)
print('Current run on Drive:')
print(DRIVE_CURRENT_RUN_DIR)
print('Best checkpoint:', DRIVE_BEST_PT, DRIVE_BEST_PT.is_file())
print('Last checkpoint:', DRIVE_LAST_PT, DRIVE_LAST_PT.is_file())
